In [1]:
# !pip install jsonschema
import json
from jsonschema import validate

from engine import Engine
import os

data_file = 'data.json'
schema_file = 'schema.json'
data_enum_error_file = 'data_enum_error.json'
data_type_error_file = 'data_type_error.json'
data_no_schema_error_file = 'data_no_schema_error.json'
data_unclosed_error_file = 'data_unclosed_error.json'

In [2]:
json_data = json.load(open(data_file))
json_schema = json.load(open(schema_file))

# Validate the JSON data against the schema using jsonschema library
# jsonschema requires valid python dict for both data and schema
validate(instance=json_data, schema=json_schema)

In [3]:
def verify_json(schema=schema_file, data=data_file, max_depth = None):
    engine = Engine(schema=schema, target=data, max_depth=max_depth)
    log_file = engine.run()
    for log in log_file:
        print(log)

In [4]:
# valid json data verification
verify_json(data=data_file)

--- Validation Error Log ---
------------------------------------------------------------------------------------------------------------------------
(maximum stack depth: 3)
--- validation done ---


In [5]:
# type error verification
verify_json(data=data_type_error_file)

--- Validation Error Log ---
------------------------------------------------------------------------------------------------------------------------
- invalid json item: path(top_object.a)
  node info: c
    <BAD VALUE>: value(c) violates schema[type(number)]
------------------------------------------------------------------------------------------------------------------------
- invalid json item: path(top_object)
  node info: {'a': invalid, 'b': valid, 'd': valid}
    <INCOMPLETE>: child(a) is not valid
------------------------------------------------------------------------------------------------------------------------
(maximum stack depth: 3)
--- validation done ---


In [6]:
# unclosed error verification
verify_json(data=data_unclosed_error_file)

--- Validation Error Log ---
------------------------------------------------------------------------------------------------------------------------
Validation incomplete: unclosed structure
- unclosed json item: path(top_object.d)
  node info: {0: valid}
    <UNCLOSED>: unclosed structure
------------------------------------------------------------------------------------------------------------------------
- unclosed json item: path(top_object)
  node info: {'a': valid, 'b': valid}
    <UNCLOSED>: unclosed structure
------------------------------------------------------------------------------------------------------------------------
(maximum stack depth: 3)
--- validation done ---


In [7]:
# maximum stack depth verification
verify_json(data=data_file, max_depth=2)

--- Validation Error Log ---
------------------------------------------------------------------------------------------------------------------------
Circuit breaker activated: Max nesting depth 2 exceeded
Validation incomplete: unclosed structure
- unclosed json item: path(top_object.b.c)
  node info: None
    <UNCLOSED>: unclosed structure
------------------------------------------------------------------------------------------------------------------------
- unclosed json item: path(top_object.b)
  node info: None
    <UNCLOSED>: unclosed structure
------------------------------------------------------------------------------------------------------------------------
- unclosed json item: path(top_object)
  node info: {'a': valid}
    <UNCLOSED>: unclosed structure
------------------------------------------------------------------------------------------------------------------------
(maximum stack depth: 2)
--- validation done ---


In [8]:
# enumeration error verification
verify_json(data=data_enum_error_file)

--- Validation Error Log ---
------------------------------------------------------------------------------------------------------------------------
- invalid json item: path(top_object.d[2])
  node info: 5
    <BAD VALUE>: value(5) violates schema[enum([3, 4])]
------------------------------------------------------------------------------------------------------------------------
- invalid json item: path(top_object.d)
  node info: {0: valid, 1: valid, 2: invalid}
    <INCOMPLETE>: value 2 not valid
------------------------------------------------------------------------------------------------------------------------
- invalid json item: path(top_object)
  node info: {'a': valid, 'b': valid, 'd': invalid}
    <INCOMPLETE>: child(d) is not valid
------------------------------------------------------------------------------------------------------------------------
(maximum stack depth: 3)
--- validation done ---


In [9]:
# unexpected json object verification
verify_json(data=data_no_schema_error_file)

--- Validation Error Log ---
------------------------------------------------------------------------------------------------------------------------
- invalid json item: path(top_object.b.e)
  node info: extra
    <UNEXPECTED>: unexpected value or object: extra
------------------------------------------------------------------------------------------------------------------------
- invalid json item: path(top_object.b)
  node info: {'c': valid, 'e': invalid}
    <INCOMPLETE>: child(e) is not valid
------------------------------------------------------------------------------------------------------------------------
- invalid json item: path(top_object)
  node info: {'a': valid, 'b': invalid, 'd': valid}
    <INCOMPLETE>: child(b) is not valid
------------------------------------------------------------------------------------------------------------------------
(maximum stack depth: 3)
--- validation done ---
